In [ ]:
import numpy as np
import pandas as pd
import gc

from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import balanced_accuracy_score
from sklearn.preprocessing import LabelEncoder

import xgboost as xgb
import lightgbm as lgb
from catboost import CatBoostClassifier

In [ ]:
train = pd.read_csv("train.csv")
test = pd.read_csv("test.csv")
TARGET = "Irrigation_Need"

MAP = {"Low":0, "Medium":1, "High":2}
INV_MAP = {0:"Low",1:"Medium",2:"High"}

y = train[TARGET].map(MAP)
train.drop(columns=[TARGET], inplace=True)

test_ids = test["id"]

In [ ]:
def feature_engineering(df):
    df = df.copy()
    
    df["temp_humidity"] = df["Temperature_C"] * df["Humidity"]
    df["rain_irrigation"] = df["Rainfall_mm"] + df["Previous_Irrigation_mm"]
    df["moisture_temp"] = df["Soil_Moisture"] * df["Temperature_C"]
    
    df["water_ratio"] = df["Previous_Irrigation_mm"] / (df["Rainfall_mm"] + 1)
    df["dryness"] = df["Temperature_C"] / (df["Humidity"] + 1)
    
    for col in ["Rainfall_mm", "Previous_Irrigation_mm", "Field_Area_hectare"]:
        df[f"log_{col}"] = np.log1p(df[col])
    
    return df

train = feature_engineering(train)
test = feature_engineering(test)

In [ ]:
cat_cols = train.select_dtypes(include="object").columns

for col in cat_cols:
    le = LabelEncoder()
    combined = pd.concat([train[col], test[col]]).astype(str)
    le.fit(combined)
    
    train[col] = le.transform(train[col].astype(str))
    test[col] = le.transform(test[col].astype(str))

In [28]:
X = train.copy().to(device="cuda")
X_test = test.copy().to(device="cuda")

y = y.reset_index(drop=True)

AttributeError: 'DataFrame' object has no attribute 'to'

In [27]:
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

oof = np.zeros((len(X), 3))
test_preds = np.zeros((len(X_test), 3))

seeds = [42, 100, 2024]

for seed in seeds:
    print(f"\n Seed {seed}")
    
    for fold, (tr_idx, val_idx) in enumerate(skf.split(X, y)):
        print(f"Fold {fold+1}")
        
        X_tr, X_val = X.iloc[tr_idx], X.iloc[val_idx]
        y_tr, y_val = y.iloc[tr_idx], y.iloc[val_idx]
        
        # XGBoost GPU (FIXED)
        xgb_model = xgb.XGBClassifier(
            n_estimators=1500,
            learning_rate=0.01,
            max_depth=6,
            subsample=0.8,
            colsample_bytree=0.8,
            objective="multi:softprob",
            num_class=3,
            eval_metric="mlogloss",
            tree_method="hist",
            device="cuda",   
            random_state=seed
        )
        
        xgb_model.fit(
            X_tr, y_tr,
            eval_set=[(X_val, y_val)],
            verbose=False
        )
        
        #  LightGBM (no warning)
        lgb_model = lgb.LGBMClassifier(
            n_estimators=1500,
            learning_rate=0.01,
            num_leaves=64,
            subsample=0.8,
            colsample_bytree=0.8,
            force_col_wise=True,   
            random_state=seed
        )
        
        lgb_model.fit(X_tr, y_tr)
        
        # CatBoost GPU
        cat_model = CatBoostClassifier(
            iterations=1500,
            learning_rate=0.03,
            depth=6,
            task_type="GPU",
            devices="0",
            verbose=0,
            random_seed=seed
        )
        
        cat_model.fit(X_tr, y_tr)
        
        #  Ensemble
        val_pred = (
            0.4 * xgb_model.predict_proba(X_val) +
            0.3 * lgb_model.predict_proba(X_val) +
            0.3 * cat_model.predict_proba(X_val)
        )
        
        test_pred = (
            0.4 * xgb_model.predict_proba(X_test) +
            0.3 * lgb_model.predict_proba(X_test) +
            0.3 * cat_model.predict_proba(X_test)
        )
        
        oof[val_idx] += val_pred / len(seeds)
        test_preds += test_pred / (len(seeds) * 5)
        print(oof[val_idx] , test_preds)

[LightGBM] [Info] Total Bins 4991
[LightGBM] [Info] Number of data points in the train set: 504000, number of used features: 28
[LightGBM] [Info] Start training from score -0.532443
[LightGBM] [Info] Start training from score -0.968948
[LightGBM] [Info] Start training from score -3.400721
[[2.02327829e-04 3.33042052e-01 8.89598264e-05]
 [9.26105002e-04 3.32371994e-01 3.52376841e-05]
 [9.38839201e-04 3.32381352e-01 1.31440897e-05]
 ...
 [9.79616809e-06 2.82204436e-03 3.30501498e-01]
 [2.43566204e-03 3.30825132e-01 7.25417889e-05]
 [1.06447807e-03 3.32095591e-01 1.73263257e-04]] [[6.66262971e-02 4.00639915e-05 3.04744599e-07]
 [5.68856469e-02 9.75249047e-03 2.85294201e-05]
 [6.66069328e-02 5.92497585e-05 4.83132280e-07]
 ...
 [5.16548127e-05 6.65508222e-02 6.41915096e-05]
 [6.50464765e-02 1.60301040e-03 1.71797497e-05]
 [1.05712849e-02 5.60420236e-02 5.33589829e-05]]
Fold 2
[LightGBM] [Info] Total Bins 4992
[LightGBM] [Info] Number of data points in the train set: 504000, number of used 

In [30]:
from sklearn.metrics import precision_score


oof_labels = oof.argmax(axis=1)
score = balanced_accuracy_score(y, oof_labels)
real_score = balanced_accuracy_score(y , oof_labels)
print("Final Balanced Accuracy:", score )

Final Balanced Accuracy: 0.9618291006416767


In [31]:
final_preds = test_preds.argmax(axis=1)

submission = pd.DataFrame({
    "id": test_ids,
    "Irrigation_Need": [INV_MAP[p] for p in final_preds]
})

submission.to_csv("submission.csv", index=False)
print("submission_acc.csv saved")

submission_acc.csv saved
